# Import

In [1]:
import os
import umap
import random
import shutil
import pickle
import numpy as np
import scanpy as sc
import tifffile as tiff
from scipy.stats import binned_statistic_2d

import torch
from torch.utils.data import Dataset,DataLoader

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.ndimage import gaussian_filter,gaussian_laplace

import warnings
warnings.filterwarnings("ignore",category=FutureWarning)

device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# For dataset

In [3]:
#skip this part when you have already created the processed file

In [12]:
def process_for_dataset(embedding_type,species,HV_GENE_NUM,USE_CELL_NUM,adata_name=None):
    if adata_name is None:
        adata_name=species
    species_data=sc.read_h5ad(f'./data/processed_data/gene_expression/{adata_name}.h5ad')
    species_data.var_names_make_unique()  
    species_embedding=torch.load(f'./data/processed_data/protein_embedding/'+\
                                f'{embedding_type}_{species}_embedding.torch',weights_only=True)

    genes_to_keep=list(species_embedding.keys())
    species_data=species_data[:,species_data.var_names.isin(genes_to_keep)].copy()

    print(f'In {species}, choose {HV_GENE_NUM} top hv genes from {len(species_data.var_names) }')
    print(f'In {species}, random choose {USE_CELL_NUM} cells from {species_data.n_obs}')

    sc.pp.normalize_total(species_data)
    sc.pp.log1p(species_data)
    sc.pp.highly_variable_genes(species_data,flavor='seurat',n_top_genes=HV_GENE_NUM,subset=True,inplace=True)

    random_indices=np.random.choice(species_data.n_obs,size=USE_CELL_NUM,replace=False)
    species_data=species_data[random_indices].copy()

    unique_categories=species_data.obs['cluster'].cat.categories
    category_to_id={category:idx for idx,category in enumerate(unique_categories)}
    print('category_to_id:')
    print(category_to_id,'\n')

    species_data.obs['cluster_id']=species_data.obs['cluster'].map(category_to_id)

    row_indices,col_indices=species_data.X.nonzero()
    indices=torch.tensor(np.array([row_indices,col_indices]),dtype=torch.int64)
    values=torch.tensor(species_data.X.data,dtype=torch.float32)
    size=torch.Size(species_data.X.shape)
    species_tensor=torch.sparse_coo_tensor(indices,values,size)

    embedding_ref_list=[species_embedding[key] for key in species_data.var_names]
    embedding_ref=torch.stack(embedding_ref_list)

    cell_names=np.array(species_data.obs_names)
    gene_names=np.array(species_data.var_names)
    labels=torch.tensor(species_data.obs['cluster_id'].values)

    process_return={'X':species_tensor,'embedding_ref':embedding_ref,'cell_names':cell_names,'gene_names':gene_names,\
                    'labels':labels,'category_to_id':category_to_id}
    return process_return

In [6]:
mission_name='Frog_Zebrafish_2000hv_60000cell'
embedding_type='ESM1b'#'ESM1b','ESM2','protXL'
all_species=['frog','zebrafish']

for species in all_species:
    print('Processing',species)
    process_return=process_for_dataset(embedding_type,species,HV_GENE_NUM=2000,USE_CELL_NUM=60000)
    mission_path=os.path.join('./use_data/Save_DataSet',mission_name)
    os.makedirs(mission_path,exist_ok=True)
    with open(os.path.join(mission_path,f'{embedding_type}_{species}_process_return.pkl'),'wb') as f:
        pickle.dump(process_return,f)

Processing frog
In frog, choose 2000 top hv genes from 9363
In frog, random choose 60000 cells from 96935
category_to_id:
{'Blastula': 0, 'Blood': 1, 'Cement gland primordium': 2, 'Endoderm': 3, 'Endothelial': 4, 'Epidermal progenitor': 5, 'Eye primordium': 6, 'Forebrain/midbrain': 7, 'Germline': 8, 'Goblet cell': 9, 'Hatching gland': 10, 'Heart': 11, 'Hindbrain': 12, 'Intermediate mesoderm': 13, 'Involuting marginal zone': 14, 'Ionocyte': 15, 'Lens': 16, 'Myeloid progenitors': 17, 'Neural crest': 18, 'Neuroectoderm': 19, 'Neuroendocrine cell': 20, 'Neuron': 21, 'Non-neural ectoderm': 22, 'Notochord': 23, 'Notoplate': 24, 'Olfactory placode': 25, 'Optic': 26, 'Otic placode': 27, 'Placodal area': 28, 'Presomitic mesoderm': 29, 'Pronephric mesenchyme': 30, 'Rohon-beard neuron': 31, 'Skeletal muscle': 32, 'Small secretory cells': 33, 'Spemann organizer': 34, 'Tailbud': 35} 

Processing zebrafish
In zebrafish, choose 2000 top hv genes from 16757
In zebrafish, random choose 60000 cells from

In [13]:
mission_name='Human_Mouse_Lemur_2000hv_15000cell'
embedding_type='ESM1b'#'ESM1b','ESM2','protXL'
all_species=['human','mouse','lemur']
adata_names=['human_blood','mouse_blood','lemur_blood']

for i,species in enumerate(all_species):
    print('Processing',species)
    process_return=process_for_dataset(embedding_type,species,HV_GENE_NUM=2000,USE_CELL_NUM=15000,adata_name=adata_names[i])
    mission_path=os.path.join('./use_data/Save_DataSet',mission_name)
    os.makedirs(mission_path,exist_ok=True)
    with open(os.path.join(mission_path,f'{embedding_type}_{species}_process_return.pkl'),'wb') as f:
        pickle.dump(process_return,f)

Processing human


D:\Working_Program\Python\Lib\site-packages\anndata\_core\anndata.py:1900: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In human, choose 2000 top hv genes from 17228
In human, random choose 15000 cells from 72403
category_to_id:
{'hematopoietic stem cell': 0, 'common myeloid progenitor': 1, 'erythrocyte': 2, 'platelet': 3, 'macrophage': 4, 'B cell': 5, 'monocyte': 6, 'natural killer cell': 7, 'CD4-positive, alpha-beta T cell': 8, 'CD8-positive, alpha-beta T cell': 9, 'basophil': 10, 'neutrophil': 11, 'plasmacytoid dendritic cell': 12, 'plasma cell': 13, 'mature NK T cell': 14, 'regulatory T cell': 15, 'classical monocyte': 16, 'non-classical monocyte': 17, 'intermediate monocyte': 18, 'hematopoietic precursor cell': 19} 

Processing mouse
In mouse, choose 2000 top hv genes from 11300
In mouse, random choose 15000 cells from 19082
category_to_id:
{'B cell': 0, 'DN1 thymic pro-T cell': 1, 'Fraction A pre-pro B cell': 2, 'T cell': 3, 'basophil': 4, 'classical monocyte': 5, 'dendritic cell': 6, 'early pro-B cell': 7, 'erythroblast': 8, 'erythrocyte': 9, 'granulocyte': 10, 'granulocytopoietic cell': 11, 'hem

# For painting Dataset

In [14]:
class SpeciesDataset(torch.utils.data.Dataset):
    def __init__(self,species_info):
        self.X=species_info['X']
        self.embedding_ref=species_info['embedding_ref']
        self.cell_names=species_info['cell_names']
        self.gene_names=species_info['gene_names']
        self.labels=species_info['labels']
        self.category_to_id=species_info['category_to_id']
        self.len=len(self.labels)

    def __getitem__(self,index):
        R_dict={}
        R_dict['cell_name']=self.cell_names[index]
        
        cell_X=self.X[index].coalesce()
        R_dict['location']=self.embedding_ref[cell_X.indices().squeeze(0)]
        R_dict['expression']=cell_X.values().unsqueeze(1)
        R_dict['label']=self.labels[index]
        return R_dict

    def __len__(self):
        return self.len

In [15]:
def reduce_location_dim(all_embeddings,dataset_cfg):
    print('Reducing location dims from',all_embeddings.shape[-1],'to',dataset_cfg['location_pca_dim'],'using PCA')
    pca_reducer=PCA(n_components=dataset_cfg['location_pca_dim'])
    embeddings_pca=pca_reducer.fit_transform(all_embeddings)
    print('pca_explained_variance_sum:',pca_reducer.explained_variance_ratio_.sum())

    if dataset_cfg['location_tsne_dim']>0:
        print('Reducing location dims from',dataset_cfg['location_pca_dim'],'to',dataset_cfg['location_tsne_dim'],'using tsne')
        tsne=TSNE(n_components=dataset_cfg['location_tsne_dim'],random_state=42,metric='cosine')
        embeddings_tsne=tsne.fit_transform(embeddings_pca)
        all_locs=embeddings_tsne
    elif dataset_cfg['location_umap_dim']>0:
        print('Reducing location dims from',dataset_cfg['location_pca_dim'],'to',dataset_cfg['location_umap_dim'],'using umap')
        umap_reducer=umap.UMAP(n_components=dataset_cfg['location_umap_dim'],n_neighbors=30,min_dist=0.3,metric='cosine')
        embeddings_umap=umap_reducer.fit_transform(embeddings_pca)
        all_locs=embeddings_umap
    else:
        all_locs=embeddings_pca
    return all_locs

In [ ]:
#let REBuild=0 when you have already created the dataset
REBuild=0

mission_name='Frog_Zebrafish_2000hv_60000cell'
embedding_type='ESM1b'
all_species=['frog','zebrafish']

dataset_cfg={'location_pca_dim':-1,'location_tsne_dim':2,'location_umap_dim':0}

paint_train_set,paint_val_set,paint_test_set={},{},{}
if REBuild==1:
    paint_dataset={}
    all_embeddings=torch.tensor([])
    for species in all_species:
        with open(f'./use_data/Save_DataSet/{mission_name}/{embedding_type}_{species}_process_return.pkl','rb') as f:
            process_return=pickle.load(f)
        paint_dataset[species]=SpeciesDataset(process_return)
        all_embeddings=torch.cat((all_embeddings,paint_dataset[species].embedding_ref),dim=0)
    if dataset_cfg['location_pca_dim']==-1:
        dataset_cfg['location_pca_dim']=all_embeddings.shape[-1]//10
    all_locs=reduce_location_dim(all_embeddings,dataset_cfg)

    loc_pointer=0
    for species in all_species:
        paint_dataset[species].embedding_ref=torch.tensor(all_locs[loc_pointer:loc_pointer+len(paint_dataset[species].embedding_ref)])
        loc_pointer+=len(paint_dataset[species].embedding_ref)

        species_train_size=int(len(paint_dataset[species])*0.6)
        species_val_size=int(len(paint_dataset[species])*0.2)
        species_test_size=len(paint_dataset[species])-species_train_size-species_val_size
        paint_train_set[species],paint_val_set[species],paint_test_set[species]=torch.utils.data.random_split(paint_dataset[species],[species_train_size,species_val_size,species_test_size])

        for paint_set,set_type in [(paint_train_set,'train'),(paint_val_set,'val'),(paint_test_set,'test')]:
            with open(f'./use_data/Save_DataSet/{mission_name}/{embedding_type}_{species}_{set_type}_set.pkl','wb') as f:
                pickle.dump(paint_set[species],f)
else:
    for species in all_species:
        for paint_set,set_type in [(paint_train_set,'train'),(paint_val_set,'val'),(paint_test_set,'test')]:
            with open(f'./use_data/Save_DataSet/{mission_name}/{embedding_type}_{species}_{set_type}_set.pkl','rb') as f:
                paint_set[species]=pickle.load(f)

In [1]:
#printed when REBuild=1 
#Reducing location dims from 1280 to 128 using PCA
#pca_explained_variance_sum: 0.8271395640827764
#Reducing location dims from 128 to 2 using tsne

#do not delete process_return file after painting, it would also be used for analysis

In [37]:
#for show
mission_name='Frog_Zebrafish_2000hv_60000cell'
embedding_type='ESM1b'
species='frog'

with open(f'./use_data/Save_DataSet/{mission_name}/{embedding_type}_{species}_process_return.pkl','rb') as f:
    process_return=pickle.load(f)
print(process_return)

{'X': tensor(indices=tensor([[    0,     0,     0,  ..., 59999, 59999, 59999],
                       [    7,    14,    20,  ...,  1946,  1974,  1989]]),
       values=tensor([0.1241, 0.4204, 0.7170,  ..., 0.7136, 0.8444, 0.6959]),
       size=(60000, 2000), nnz=10728073, layout=torch.sparse_coo), 'embedding_ref': tensor([[-0.0251,  0.3375,  0.0384,  ...,  0.1371, -0.0058,  0.0354],
        [ 0.0409,  0.2018, -0.0435,  ..., -0.0816, -0.1320,  0.2452],
        [ 0.0111,  0.1850, -0.1094,  ..., -0.0051, -0.0740,  0.0560],
        ...,
        [-0.0323,  0.0567, -0.0489,  ..., -0.0973, -0.0689, -0.0110],
        [ 0.0076,  0.1169, -0.0643,  ..., -0.0617,  0.0089, -0.0550],
        [-0.0418,  0.1353,  0.2108,  ..., -0.1011, -0.1472,  0.0236]]), 'cell_names': array(['AACGGTAGC-ACGCCATT', 'CCGATACG-TGCAAGGG', 'ACTGAGTGC-TACCAGGC',
       ..., 'CCATCCAC-ACATCTAT-1', 'ACGCTCTCA-ACCCATAT',
       'AAGCTACGG-GGTCACAG'], dtype=object), 'gene_names': array(['42Sp43', '42Sp50', 'MGC107908', ..., 'z

In [17]:
#let REBuild=0 when you have already created the dataset
REBuild=0

mission_name='Human_Mouse_Lemur_2000hv_15000cell'
embedding_type='ESM1b'
all_species=['human','mouse','lemur']

dataset_cfg={'location_pca_dim':-1,'location_tsne_dim':2,'location_umap_dim':0}

paint_train_set,paint_val_set,paint_test_set={},{},{}
if REBuild==1:
    paint_dataset={}
    all_embeddings=torch.tensor([])
    for species in all_species:
        with open(f'./use_data/Save_DataSet/{mission_name}/{embedding_type}_{species}_process_return.pkl','rb') as f:
            process_return=pickle.load(f)
        paint_dataset[species]=SpeciesDataset(process_return)
        all_embeddings=torch.cat((all_embeddings,paint_dataset[species].embedding_ref),dim=0)
    if dataset_cfg['location_pca_dim']==-1:
        dataset_cfg['location_pca_dim']=all_embeddings.shape[-1]//10
    all_locs=reduce_location_dim(all_embeddings,dataset_cfg)

    loc_pointer=0
    for species in all_species:
        paint_dataset[species].embedding_ref=torch.tensor(all_locs[loc_pointer:loc_pointer+len(paint_dataset[species].embedding_ref)])
        loc_pointer+=len(paint_dataset[species].embedding_ref)

        species_train_size=int(len(paint_dataset[species])*0.6)
        species_val_size=int(len(paint_dataset[species])*0.2)
        species_test_size=len(paint_dataset[species])-species_train_size-species_val_size
        paint_train_set[species],paint_val_set[species],paint_test_set[species]=torch.utils.data.random_split(paint_dataset[species],[species_train_size,species_val_size,species_test_size])

        for paint_set,set_type in [(paint_train_set,'train'),(paint_val_set,'val'),(paint_test_set,'test')]:
            with open(f'./use_data/Save_DataSet/{mission_name}/{embedding_type}_{species}_{set_type}_set.pkl','wb') as f:
                pickle.dump(paint_set[species],f)
else:
    for species in all_species:
        for paint_set,set_type in [(paint_train_set,'train'),(paint_val_set,'val'),(paint_test_set,'test')]:
            with open(f'./use_data/Save_DataSet/{mission_name}/{embedding_type}_{species}_{set_type}_set.pkl','rb') as f:
                paint_set[species]=pickle.load(f)

# Painting

In [19]:
def paint(root_path,embedding_type,all_locs,paint_list=None,with_gaussian_filter=True):    
    margin_ratio=0.1
    x_min,x_max=all_locs[:,0].min(),all_locs[:,0].max()
    y_min,y_max=all_locs[:,1].min(),all_locs[:,1].max()
    x_margin=(x_max-x_min)*margin_ratio
    y_margin=(y_max-y_min)*margin_ratio
    x_min,x_max=x_min-x_margin,x_max+x_margin
    y_min,y_max=y_min-y_margin,y_max+y_margin

    grid_size=512
    x_bins=np.linspace(x_min,x_max,grid_size+1)
    y_bins=np.linspace(y_min,y_max,grid_size+1)

    embed_channel=np.zeros((1,grid_size,grid_size),dtype=np.float32)
    embed_channel[0],_,_,_=binned_statistic_2d(all_locs[:,0],all_locs[:,1],values=None,statistic='count',bins=[x_bins,y_bins])
    embed_channel=np.log1p(embed_channel)

    pos_channels=np.zeros((2,grid_size,grid_size),dtype=np.float32)
    x_centers=(x_bins[:-1]+x_bins[1:])/2
    y_centers=(y_bins[:-1]+y_bins[1:])/2
    pos_channels[0]=x_centers.reshape(1,-1)
    pos_channels[1]=y_centers.reshape(-1,1)
    
    sigma_values=[1,3,5]
    fuse_weights=[0.5,0.3,0.2]
    if with_gaussian_filter is True:
        embed_channel=sum(w*gaussian_filter(embed_channel,sigma=s) for w,s in zip(fuse_weights,sigma_values))
    tiff.imwrite(f'{root_path}/{embedding_type}_background_emb.tiff',embed_channel,compression='zlib')
    tiff.imwrite(f'{root_path}/{embedding_type}_background_pos.tiff',pos_channels,compression='zlib')

    for paint_material,folder_path,serial_number in paint_list:
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
        px_indices=np.digitize(paint_material['location'][:,0],bins=x_bins)-1
        py_indices=np.digitize(paint_material['location'][:,1],bins=y_bins)-1

        expression_channel=np.zeros((1,grid_size,grid_size),dtype=np.float32)
        keep_dict={}
        for i in range(len(paint_material['expression'])):
            px,py=px_indices[i],py_indices[i]
            if (px,py) not in keep_dict:
                keep_dict[(px,py)]=[]
            keep_dict[(px,py)].append(paint_material['expression'][i])
        for (px,py),expressions in keep_dict.items():
            expression_channel[0,px,py]=np.sqrt(np.sum(np.square(expressions)))
        sigma_values=[1,3,5]
        fuse_weights=[0.5,0.3,0.2]
        if with_gaussian_filter is True:
            expression_channel=sum(w*gaussian_filter(expression_channel,sigma=s) for w,s in zip(fuse_weights,sigma_values))
        tiff.imwrite(f'{folder_path}/{serial_number}.tiff',expression_channel)#,compression=
        #compression would decrease file size 2 times, while increase training time 2 times

In [20]:
def prepare_for_sample_painting(root_path,embedding_type,all_species):
    all_locs=torch.tensor([])
    cell_corr_list,class_corr_list=[],[]
    paint_list=[]
    serial_number=0
    for species in all_species:
        del_path=f'{root_path}/{embedding_type}_{species}'
        if os.path.exists(del_path):
            try:
                shutil.rmtree(del_path)
            except Exception as e:
                print(f'Folder deleting error: {e}')
        os.makedirs(del_path)
    
        all_locs=torch.cat((all_locs,paint_train_set[species].dataset.embedding_ref),dim=0)
    
        for paint_set,set_type in [(paint_train_set,'train'),(paint_val_set,'val'),(paint_test_set,'test')]:
            for paint_material in paint_set[species]:
                label=paint_material['label'].item()
                paint_list.append((paint_material,f'{root_path}/{embedding_type}_{species}/{set_type}/{label}',serial_number))
                cell_corr_list.append((serial_number,species,paint_material['cell_name']))
                serial_number+=1
        for cell_type,_class in paint_train_set[species].dataset.category_to_id.items():
            class_corr_list.append((species,_class,cell_type))

    with open(f'{root_path}/{embedding_type}_image_cell_correspondence.txt','w') as f:
        for serial_number,species,cell_name in cell_corr_list:
            f.write(f'{serial_number}\t{species}\t{cell_name}\n')
    with open(f'{root_path}/{embedding_type}_category_label_correspondence.txt','w') as f:
        for species,_class,cell_type in class_corr_list:
            f.write(f'{species}\t{_class}\t{cell_type}\n')
    return all_locs,paint_list

In [10]:
# Painting frog and zebrafish
# It may take some time
mission_name='Frog_Zebrafish_2000hv_60000cell'
embedding_type='ESM1b'
all_species=['frog','zebrafish']

root_path=f'./use_data/Heatmap_Paintings/{mission_name}'

print('preparing for sample painting')
all_locs,paint_list=prepare_for_sample_painting(root_path,embedding_type,all_species)

print('begin painting')
paint(root_path,embedding_type,all_locs,paint_list)
print('done painting')

preparing for sample painting
begin painting
done painting


In [ ]:
# 117 GB for 60000 painted cells

In [21]:
# Painting human, mouse and lemur
# It may take some time
mission_name='Human_Mouse_Lemur_2000hv_15000cell'
embedding_type='ESM1b'
all_species=['human','mouse','lemur']

root_path=f'./use_data/Heatmap_Paintings/{mission_name}'

print('preparing for sample painting')
all_locs,paint_list=prepare_for_sample_painting(root_path,embedding_type,all_species)

print('begin painting')
paint(root_path,embedding_type,all_locs,paint_list)
print('done painting')

preparing for sample painting
begin painting
done painting


# For Ablation study

In [9]:
# Painting frog and zebrafish without gaussian filter
mission_name='Frog_Zebrafish_2000hv_60000cell_without_gaussian_filter'##
#the datasets used here are still created under mission_name: Frog_Zebrafish_2000hv_60000cell
#only the saved path and paint function is changed here
embedding_type='ESM1b'
all_species=['frog','zebrafish']

root_path=f'./use_data/Heatmap_Paintings/{mission_name}'

print('preparing for sample painting')
all_locs,paint_list=prepare_for_sample_painting(root_path,embedding_type,all_species)

print('begin painting')
paint(root_path,embedding_type,all_locs,paint_list,with_gaussian_filter=False)
print('done painting')

preparing for sample painting
begin painting
done painting
